# Notebook 3: Serve & Predict

You've explored the data (notebook 1) and trained a model (notebook 2). Now let's put it to work.

**What you'll do in this notebook:**

1. Load the trained model from MLflow
2. Run predictions on sample texts
3. Understand how the FastAPI serving endpoint works

**What you'll learn:**
- How MLflow stores and loads models
- How to go from a trained model to a prediction in just a few lines
- How a model gets served as a web API that other applications can call

---

### Notebook series

| Notebook | Focus |
|---|---|
| 01 — Setup & EDA | Environment, data loading, exploration |
| 02 — Train & Evaluate | Build, train, and evaluate a text classifier |
| **03 — Serve & Predict** (you are here) | Load the trained model and make predictions |

---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

import mlflow
import mlflow.pyfunc

# Locate the project root
PROJECT_ROOT = (
    Path("__vsc_ipynb_file__").resolve().parent.parent
    if "__vsc_ipynb_file__" in dir()
    else Path.cwd().parent
)
if not (PROJECT_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import get_env

# Point MLflow at our local tracking directory
tracking_uri = get_env("MLFLOW_TRACKING_URI", "./mlruns")
mlflow.set_tracking_uri(str(PROJECT_ROOT / tracking_uri))

print(f"Project root: {PROJECT_ROOT}")

---
## 2. Load the Trained Model

In notebook 2, we saved our trained model as an MLflow artifact and wrote the run ID to a file. Now we'll load it back.

### How MLflow model loading works

MLflow uses a URI format to locate models: `runs:/<run_id>/model`. This tells MLflow:
- Look in the run with this ID
- Find the artifact named "model"
- Deserialize it back into a working Python object

We use `mlflow.pyfunc.load_model()` which returns a generic "PyFunc" wrapper. This is the same interface the FastAPI server uses — so what you see here is exactly how the API serves predictions.

In [ ]:
# Read the run ID saved by notebook 2
run_id_path = PROJECT_ROOT / "data" / "interim" / "latest_run_id.txt"
RUN_ID = run_id_path.read_text().strip()
print(f"Loading model from MLflow run: {RUN_ID}")

# Load the model
model_uri = f"runs:/{RUN_ID}/model"
loaded_model = mlflow.pyfunc.load_model(model_uri)

print(f"Model loaded successfully from: {model_uri}")
print(f"Model type: {type(loaded_model)}")

---
## 3. Make Predictions

Let's test the model on some made-up headlines to see if it can correctly categorize them. These are texts the model has never seen before — a simple "sanity check" that the model works as expected.

The AG News categories are:

| Label | Category | Example topics |
|---|---|---|
| 0 | World | International politics, diplomacy, conflicts |
| 1 | Sports | Games, scores, athletes, tournaments |
| 2 | Business | Markets, earnings, mergers, economy |
| 3 | Sci/Tech | Research, gadgets, software, space |

In [ ]:
# Some sample headlines spanning all four categories
sample_texts = [
    "NASA launches new spacecraft to explore Mars",
    "Stock market hits record high amid economic optimism",
    "Champions League final draws record viewership",
    "Google unveils latest advances in artificial intelligence",
    "UN Security Council holds emergency meeting on conflict",
    "Lakers defeat Celtics in overtime thriller",
]

label_names = {0: "World", 1: "Sports", 2: "Business", 3: "Sci/Tech"}

# Run predictions
predictions = loaded_model.predict(sample_texts)

print("Predictions:\n")
for text, pred in zip(sample_texts, predictions):
    label_int = int(pred)
    category = label_names.get(label_int, str(label_int))
    print(f"  [{category:10s}]  {text}")

### Try your own texts

Edit the cell below to test the model on any text you want. Does it get them right? Can you find examples that trick it?

In [ ]:
# Replace this with any text you want to classify
your_text = "Apple announces record quarterly revenue driven by iPhone sales"

pred = loaded_model.predict([your_text])
label_int = int(pred[0])
print(f"Text:     {your_text}")
print(f"Category: {label_names[label_int]} (label {label_int})")

---
## 4. How the FastAPI Endpoint Works

In a real application, you wouldn't open a notebook every time you want a prediction. Instead, you'd serve the model as a **web API** — a URL that other programs can send text to and get predictions back.

This project includes a FastAPI server in `src/serve.py` that does exactly what we just did above, but over HTTP.

### The key idea

The serving code is surprisingly short:

```python
# 1. Load the model once at startup
model = mlflow.pyfunc.load_model(model_path)

# 2. For each incoming request, run a prediction
@app.post("/infer")
def infer(payload: InferRequest):
    pred = model.predict([payload.text])
    return {"label": int(pred[0])}
```

That's it — the same `model.predict()` call we used above, wrapped in a web endpoint.

### How to run it

From the project root (not from this notebook), run:

```bash
uvicorn src.serve:app --reload
```

Then send it a request:

```bash
curl -X POST http://localhost:8000/infer \
     -H "Content-Type: application/json" \
     -d '{"text": "NASA launches new spacecraft to explore Mars"}'
```

You should get back something like:
```json
{"label": 3}
```

### What `--reload` does

The `--reload` flag tells uvicorn to watch for file changes and automatically restart the server. This is handy during development but should be turned off in production.

### Why FastAPI?

FastAPI is a popular Python web framework that's:
- **Fast** — built on async Python for high throughput
- **Self-documenting** — automatically generates API docs at `/docs`
- **Type-safe** — uses Pydantic for request/response validation

---
## 5. Look at the Source Code

Throughout these notebooks, we inlined all the logic so you could see every step. But the project also has equivalent module files in `src/` that are ready for production use:

| Module | What it does | Notebook equivalent |
|---|---|---|
| `src/data.py` | Loads HF datasets into DataFrames | Notebook 1, section 3 |
| `src/eda.py` | Prints stats and saves plots | Notebook 1, section 4 |
| `src/baseline.py` | Trains TF-IDF + LogReg with MLflow | Notebook 2, sections 3–5 |
| `src/serve.py` | FastAPI inference endpoint | This notebook, section 4 |
| `src/utils.py` | `set_all_seeds()` and `get_env()` helpers | Used everywhere |

You can run these from the terminal as modules:

```bash
python -m src.eda          # Run EDA
python -m src.baseline     # Train the model
uvicorn src.serve:app      # Serve predictions
```

The notebooks are for learning and exploration; the `src/` modules are for repeatable, scriptable workflows.

---
## Summary

Here's what we accomplished across all three notebooks:

1. **Setup & EDA** — Installed dependencies, loaded AG News, explored the data
2. **Train & Evaluate** — Built a TF-IDF + Logistic Regression pipeline, trained it, and evaluated with a confusion matrix and classification report
3. **Serve & Predict** — Loaded the saved model, made predictions, and learned how the FastAPI endpoint works

### What to explore next

- **Try a different dataset** — Change `dataset` in `configs/baseline.yaml` to `imdb`, `rotten_tomatoes`, or another HF text classification dataset
- **Tune hyperparameters** — Adjust `C`, `max_iter`, `max_features`, or `ngram_range` and compare runs in MLflow
- **Swap the model** — Replace `LogisticRegression` with `SGDClassifier` or `RandomForestClassifier`
- **Launch the API** — Run `uvicorn src.serve:app --reload` and try it with real queries
- **Browse MLflow** — Run `mlflow ui --backend-store-uri ./mlruns` and explore your experiment history

Happy hacking!